#### [ ]


- 구현
    - 입력 텍스트의 길이를 측정해서 길이만큼 추출후 진행

In [14]:
import torch
import torch.nn as nn


from torchtext.datasets import AG_NEWS as AG_NEWS


In [15]:
## 데이터준비 
SAVE_DIR ='../_data/'


[2] 데이터 로딩 및 확인 <hr>

In [16]:
## pytorch의 torchtext의 내장 데이터셋 저장
trainDS, testDS = AG_NEWS(SAVE_DIR)

In [17]:
for a, b in trainDS:
    print(a, b)
    break

3 Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.


In [18]:
# [3] 데이터로더 생성 <hr>
from torch.utils.data import DataLoader
## 데이터셋, 데이터로더 관련 모듈

from torch.nn.utils.rnn import pad_sequence
## 데이터 길이 맞추기

## 토커나이저, 단어사전 관련 모듈
from torchtext.data.utils import get_tokenizer              ## 토커나이저 인스턴스 추출
from torchtext.vocab import build_vocab_from_iterator       ## 데이터셋에서 단어사전 생성 함수
from nltk.corpus import stopwords                           ## 불용어 데이터셋
import nltk

In [19]:
print(nltk.__file__)

c:\Users\kdt\anaconda3\envs\NLP\lib\site-packages\nltk\__init__.py


In [20]:
### 특별문자토큰
UNK, PAD = '<UNK>', '<PAD>'
STOPWORDS = stopwords.words('english')
print(f'Stopwords : {STOPWORDS[:10]}')

### ==> 토커나이즈 생성
tokenizer = get_tokenizer('basic_english')


Stopwords : ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an']


In [23]:
## 토큰 제네레이터 함수 : 데이터 추출하여 토큰화
def yield_tokens(data_iter):
    for label, news in data_iter:
        #라벨 텍스트 
        tokens = tokenizer(news)
        print(f"[tokens 1] =>  {len(tokens)}")
        tokens = [token for token in tokens if token not in STOPWORDS]
        print(f"[tokens 2] =>  {len(tokens)}")
        yield tokens

In [24]:
vocab = build_vocab_from_iterator(yield_tokens(trainDS),
                                    specials= [PAD, UNK],
                                    special_first=True
)




[tokens 1] =>  29
[tokens 2] =>  23
[tokens 1] =>  42
[tokens 2] =>  32
[tokens 1] =>  40
[tokens 2] =>  29
[tokens 1] =>  40
[tokens 2] =>  32
[tokens 1] =>  43
[tokens 2] =>  37
[tokens 1] =>  48
[tokens 2] =>  39
[tokens 1] =>  47
[tokens 2] =>  38
[tokens 1] =>  49
[tokens 2] =>  39
[tokens 1] =>  88
[tokens 2] =>  61
[tokens 1] =>  31
[tokens 2] =>  25
[tokens 1] =>  44
[tokens 2] =>  30
[tokens 1] =>  56
[tokens 2] =>  44
[tokens 1] =>  31
[tokens 2] =>  26
[tokens 1] =>  50
[tokens 2] =>  36
[tokens 1] =>  54
[tokens 2] =>  39
[tokens 1] =>  32
[tokens 2] =>  18
[tokens 1] =>  26
[tokens 2] =>  15
[tokens 1] =>  52
[tokens 2] =>  29
[tokens 1] =>  32
[tokens 2] =>  24
[tokens 1] =>  31
[tokens 2] =>  23
[tokens 1] =>  29
[tokens 2] =>  21
[tokens 1] =>  28
[tokens 2] =>  19
[tokens 1] =>  22
[tokens 2] =>  14
[tokens 1] =>  34
[tokens 2] =>  24
[tokens 1] =>  30
[tokens 2] =>  20
[tokens 1] =>  33
[tokens 2] =>  24
[tokens 1] =>  31
[tokens 2] =>  20
[tokens 1] =>  28
[tokens 2]

In [25]:
vocab.set_default_index(0)

In [26]:
# 텍스트 >> 정수 인코딩
text_pipeline = lambda x : vocab(tokenizer(x))

## 레이블 >> 정수 인코딩 (0~3)
label_pipeline = lambda x: int(x) -1

In [27]:
# [3-2] 단어사전 생성
## 배치크기만큼 데이터 로딩 시 위치 지정 위한 설정
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [37]:
## 함수기능 : 배치크기 만큼 데이터셋 로딩해서 토큰+ 텐서화 진행 후 반환
def collate_batch(batch):
    label_list, news_list, offsets = [], [], [0]
    
     # 1개씩 뉴스기사, 라벨 추출 해서 저장 
    for _label, _news in batch:
         # 라벨 인코딩 후 저장
         label_list.append(label_pipeline(_label))
         
         # 텍스트 인코딩 후 저장
         processed_news = torch.tensor(text_pipeline(_news), dtype=torch.int64)
         news_list.append(processed_news)
         
         # 다음 뉴스를 읽기 위한 위치값 정보
         offsets.append(processed_news.size(0))
         
    
    # 배치 크기 만큼의 라벨 리스트 => 텐서화 진행     
    label_list = torch.tensor(label_list, dtype=torch.float32)
    
    # 배치 크기 만큼의 길이 위치값 => 텐서화
    offsets = torch.tensor(offsets[:-1]).cumsum(dim=0)
    # 배치 크기 만큼의 뉴스 기사 리스트 => 텐서화
    news_list = torch.cat(news_list)
    
    # # 문장의 길이 일치
    # news_list = pad_sequence(news_list, batch_first=True, padding_value=0)
    
    return label_list.to(DEVICE), news_list.to(DEVICE), offsets.to(DEVICE)

In [38]:
# DL 생성

BATCH_SIZE = 5

trainDL = DataLoader( trainDS,
                     batch_size=BATCH_SIZE,
                     shuffle=True,
                     collate_fn=collate_batch)

testDL = DataLoader( testDS,
                    batch_size=BATCH_SIZE,
                     shuffle=True,
                     collate_fn=collate_batch)

In [41]:
# for a, b in trainDS:
#     print(a,b)

In [42]:
# HIDDEN_SIZE = 3
# EMBEDD_DIM = 10
# VOCAB_SIZE = len(vocab)

cnt = 5
for label, news, offset in trainDL:
    print(f"\nlabels : {label.shape} => {label}")
    print(f"news     : {news.shape}")
    print(f"offset  : {offset}")
    cnt -= 1
    if cnt ==0: break


labels : torch.Size([5]) => tensor([1., 3., 1., 2., 3.])
news     : torch.Size([218])
offset  : tensor([  0,  43,  90, 143, 184])

labels : torch.Size([5]) => tensor([2., 3., 2., 0., 1.])
news     : torch.Size([250])
offset  : tensor([  0,  41,  93, 137, 196])

labels : torch.Size([5]) => tensor([3., 0., 2., 2., 0.])
news     : torch.Size([212])
offset  : tensor([  0,  33,  68, 125, 164])

labels : torch.Size([5]) => tensor([1., 0., 2., 1., 2.])
news     : torch.Size([219])
offset  : tensor([  0,  44,  84, 148, 175])

labels : torch.Size([5]) => tensor([1., 3., 1., 1., 1.])
news     : torch.Size([198])
offset  : tensor([  0,  52,  88, 125, 161])


[4] 모델 클래스 정의 및 설계 <hr>

In [43]:
## 클래스이름 :
## 부모클래스 :
## -----------------------
class TextModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        
        #고차원 저차원
        self.embedding = nn.EmbeddingBag(vocab_size, embed_dim, sparse=False)
        
        # 다중분류
        self.fc = nn.Linear(embed_dim, num_class)
        #초기 가중치 => self.메서드이름()
        self.init_weights()
    
    ## 가중치 초기화 기능의 메서드    
    def init_weights(self):
        initrange = 0.5
        self.embedding.weight.data.uniform_(-initrange, initrange)
        self.fc.weight.data.uniform_(-initrange, initrange)
        self.fc.bias.data.zero_()
        
    ## 전방향 학습 메서드
    def forward(self, text, offsets):
        ## 배치 크기만큼 학습 데이터 전달
        ## 임베딩층 ==> 학습하는게 아니고 차원축소만함.
        embedded = self.embedding(text, offsets)
        return  self.fc(embedded)       ## 다중분류로 손실함수에서 softmax() 처리

In [ ]:
# !python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     --------------------- ------------------ 6.8/12.8 MB 35.0 MB/s eta 0:00:01
     ----------------------- ---------------- 7.6/12.8 MB 33.5 MB/s eta 0:00:01
     ----------------------- ---------------- 7.6/12.8 MB 33.5 MB/s eta 0:00:01
     ----------------------- ---------------- 7.6/12.8 MB 33.5 MB/s eta 0:00:01
     ----------------------- ---------------- 7.6/12.8 MB 33.5 MB/s eta 0:00:01
     --------------------------- ------------ 8.7/12.8 MB 6.7 MB/s eta 0:00:01
     --------------------------- ------------ 8.9/12.8 MB 6.1 MB/s eta 0:00:01
     --------------------------- ------------ 8.9/12.8 MB 6.1 MB/s eta 0:00:01
     ---------------------------- ----------- 9.2/12.8 MB 4.8 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 6.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [45]:
## 단어사전 만들기
from torchtext.vocab import build_vocab_from_iterator
import spacy 
import string

In [46]:
PUNC    = string.punctuation + "★"
PUNC

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~★'

In [51]:
### ===> 토큰관련 특별 문자
UNK = '<UNK>'
PAD = '<PAD>'
### 토큰화 인스턴스 생성
tokenizer = spacy.load("en_core_web_sm")
### ===> 토큰 제너레이터 함수 : 데이터 추출하여 토큰화 
# def yield_tokens(data_iter):
#     count= 0
#     for label, text in data_iter:
#         # 라벨, 텍스트 --> 텍스트 토큰화
#         for i in text:
#             if i in PUNC: 
#                 text = text.replace(i,'')
#         print(text)
#         yield tokenizer(text)  ## 메모리 관리
#         count +=1
#         if count == 5:
#             break
        
        
def yield_tokens(data_iter):
    count = 0
    for label, text in data_iter:
        # 특수문자 제거
        text = ''.join([ch for ch in text if ch not in PUNC])
        
        # 토큰화 후 토큰 문자열 리스트 반환
        tokens = [tok.text for tok in tokenizer(text)]
        yield tokens
        
        # count += 1
        # if count == 5:
        #     break


In [52]:
a= yield_tokens(trainDS)
print( a)

<generator object yield_tokens at 0x00000249061683C0>


In [53]:
### ===> 토큰화 및 단어/어휘 사전 생성
VOCAB = build_vocab_from_iterator(
    yield_tokens(trainDS),
    min_freq=2,
    specials= [PAD, UNK],
    special_first=True
)

### <UNK> 인덱스 설정
VOCAB.set_default_index(VOCAB[UNK])

In [55]:
VOCAB.get_itos()[:]

['<PAD>',
 '<UNK>',
 'the',
 'to',
 'a',
 'of',
 ' ',
 'in',
 'and',
 'on',
 'for',
 '39s',
 'that',
 'The',
 'with',
 'as',
 'at',
 'its',
 'is',
 'said',
 'by',
 'US',
 'has',
 'Reuters',
 'from',
 'it',
 'AP',
 'an',
 'his',
 'will',
 'was',
 'after',
 'be',
 'have',
 'new',
 'their',
 'over',
 'are',
 'A',
 '39',
 'first',
 'up',
 'he',
 'more',
 'but',
 'two',
 'Monday',
 'Wednesday',
 'Tuesday',
 'Thursday',
 'this',
 'New',
 'Friday',
 'company',
 'Inc',
 'out',
 'against',
 'not',
 'into',
 'than',
 'about',
 'yesterday',
 'last',
 'Iraq',
 'who',
 'were',
 'one',
 'they',
 'year',
 'Microsoft',
 'been',
 'had',
 'million',
 'United',
 'Corp',
 'years',
 'would',
 'Sunday',
 'week',
 'could',
 'which',
 'AFP',
 'oil',
 'quot',
 'people',
 'today',
 'prices',
 'government',
 'President',
 'says',
 'when',
 'three',
 'percent',
 'time',
 'NEW',
 'Saturday',
 'world',
 'next',
 'or',
 'game',
 'night',
 'off',
 'YORK',
 'software',
 'back',
 'win',
 'season',
 'China',
 'World',
 